# Lean Statement Diversity Dataset — Perturbation Pipeline
Rule-based transforms — no API key required. Produces `(anchor, variant, transformation_type)` triples.

In [ ]:
import re, json, math, subprocess, tempfile, os, sys
import pandas as pd
from pathlib import Path
from huggingface_hub import HfFileSystem
from itertools import permutations
from enum import Enum
from typing import Optional
import random

# Walk up to the Lake project root (the directory containing Wiggle.lean),
# then put src/ on sys.path so we can import the perturbation library.
PROJECT_DIR = Path.cwd()
while PROJECT_DIR != PROJECT_DIR.parent and not (PROJECT_DIR / "Wiggle.lean").exists():
    PROJECT_DIR = PROJECT_DIR.parent
sys.path.insert(0, str(PROJECT_DIR / "src"))

from bounds import flip_bound, perturb_bound
from typeclass_mutate import (
    weaken_hypothesis_typeclass,
    strengthen_hypothesis_typeclass,
    strengthen_conclusion_typeclass,
    weaken_conclusion_typeclass,
)

fs = HfFileSystem()
indices = [5145, 181597, 196429, 14831, 101254, 172154, 44743, 60309, 44790, 63001]

OUTPUT_FILE = PROJECT_DIR / "pipeline" / "perturbation_pairs.jsonl"

In [13]:
df = pd.read_json(
    "hf://datasets/FrenzyMath/mathlib_informal_v4.19.0/data.jsonl",
    lines=True,
)

In [14]:
# df = df.iloc[indices]
df = df[0:10]

df = df[['signature', 'type']]
df.head()

,signature,type
0,(A : Hopf_ C) :\n A.X.comul.hom ≫\n A.X...,∀ {C : Type u₁} [inst : CategoryTheory.Categor...
1,{W : C} (k : Y ⟶ W) (h : f ≫ k = g ≫ k) : ∃! ...,∀ {C : Type u} [inst : CategoryTheory.Category...
2,{P : C} (ι : P ⟶ X) (w : ι ≫ f = ι ≫ g) : (Fo...,∀ {C : Type u} [inst : CategoryTheory.Category...
3,{P : C} {ι ι' : P ⟶ X} (w : ι ≫ f = ι ≫ g) (w...,{C : Type u} →\n [inst : CategoryTheory.Categ...
4,(f g : X ⟶ Y) : (parallelPair f g).obj one = Y,∀ {C : Type u} [inst : CategoryTheory.Category...


In [15]:
# ── Lean runner ───────────────────────────────────────────────────────────────
# PROJECT_DIR was resolved to the Lake project root in the imports cell above.
_TMP_LEAN = str(PROJECT_DIR / "_tmp_nb.lean")

def _run_lean(code: str) -> str:
    try:
        with open(_TMP_LEAN, "w") as f:
            f.write(code)
        result = subprocess.run(
            ["lake", "env", "lean", _TMP_LEAN],
            cwd=PROJECT_DIR,
            capture_output=True,
            text=True,
            timeout=300,
        )
        return result.stdout + result.stderr
    finally:
        if os.path.exists(_TMP_LEAN):
            os.remove(_TMP_LEAN)

def compile_lean(variant_sig: str, variant_type: str) -> bool:
    output = _run_lean(f"import Mathlib\n\nexample : {variant_type} := by sorry\n")
    return "error:" not in output


In [ ]:
# ── is_true ───────────────────────────────────────────────────────────────────
# Canonical truth-propagation rules. Names MUST match the keys in TRANSFORMS
# below and the perturbation names used in hackathon-demo/demo.ipynb so that
# records produced by either pipeline are interchangeable.
_PROPAGATION_RULES = {
    # Logical structure (Lean tactics)
    "negate":             {"true": "false",   "false": "true",    "unknown": "unknown"},
    "contrapose":         {"true": "true",    "false": "false",   "unknown": "unknown"},
    "converse":           {"true": "unknown", "false": "unknown", "unknown": "unknown"},
    # Hypothesis removal (Lean tactic; only removes Prop hyps not used in goal)
    "drop_unused_hyp":    {"true": "true",    "false": "unknown", "unknown": "unknown"},
    # Typeclass mutations (Python text substitution, Lean type-check oracle).
    # Names describe what happens to the TYPECLASS at that position.
    "tc_weaken_hyp":      {"true": "unknown", "false": "false",   "unknown": "unknown"},
    "tc_strengthen_hyp":  {"true": "true",    "false": "unknown", "unknown": "unknown"},
    "tc_strengthen_conc": {"true": "unknown", "false": "false",   "unknown": "unknown"},
    "tc_weaken_conc":     {"true": "true",    "false": "unknown", "unknown": "unknown"},
    # Bound mutations (regex on the type string)
    "flip_bound":         {"true": "unknown", "false": "unknown", "unknown": "unknown"},
    "bound_tighter":      {"true": "unknown", "false": "false",   "unknown": "unknown"},
}

def is_true(perturbation_path: list[str], variant_sig: str, variant_type: str) -> str:
    """
    Propagate truth symbolically through the perturbation path.
    Anchors are always 'true' (they come from Mathlib — all proven theorems).
    Returns one of: "true", "false", "unknown".

    Unknown perturbation names silently propagate to "unknown" — make sure
    new perturbations are added to _PROPAGATION_RULES above before use.
    """
    current = "true"
    for perturbation in perturbation_path:
        rules = _PROPAGATION_RULES.get(perturbation)
        current = rules[current] if rules else "unknown"
    return current

In [ ]:
# ── Perturbation functions ────────────────────────────────────────────────────
# Each perturbation: (sig, type_str) -> (variant_sig, variant_type) | None.
# Names below match the keys in _PROPAGATION_RULES and the perturbation names
# used in hackathon-demo/demo.ipynb so output records are interchangeable.

def _extract_goal(tactic: str, type_str: str) -> tuple[str, str] | None:
    """
    Shared helper: runs a Lean snippet with the given tactic, scrapes the
    extract_goal output, and returns (variant_sig, variant_type).
    """
    output = _run_lean(
        f"import Mathlib\nimport Wiggle\n\nexample : {type_str} := by\n  {tactic}\n  extract_goal\n  sorry\n"
    )
    m = re.search(r"^theorem .*extracted.*$", output, re.MULTILINE)
    if m is None:
        return None
    full_statement = m.group(0)
    without_proof = full_statement.rsplit(":= sorry", 1)[0].strip()
    parts = without_proof.split(" : ", 1)
    if len(parts) != 2:
        return None
    variant_sig, variant_type = parts
    return variant_sig.strip(), variant_type.strip()


# ── Lean-tactic perturbations ─────────────────────────────────────────────────
def negate(sig: str, type_str: str) -> tuple[str, str] | None:
    return _extract_goal("negate_state", type_str)


def contrapose(sig: str, type_str: str) -> tuple[str, str] | None:
    return _extract_goal("contrapositive", type_str)


def converse(sig: str, type_str: str) -> tuple[str, str] | None:
    return _extract_goal("converse", type_str)


def drop_unused_hyp(sig: str, type_str: str) -> tuple[str, str] | None:
    # Removes all Prop-valued hypotheses that are not used in the goal.
    # (Earlier versions of this notebook called a non-existent
    # `generalize_state` tactic — the actual Wiggle tactic is drop_unused_hyp.)
    return _extract_goal("drop_unused_hyp", type_str)


# ── Typeclass-mutation perturbations (wrap library calls, skip no-ops) ───────
def _tc_wrap(fn, sig: str, type_str: str) -> tuple[str, str] | None:
    result = fn(sig, type_str)
    if result is None:
        return None
    variant_sig, variant_type = result
    if (variant_sig.strip() == sig.strip()
            and variant_type.strip() == type_str.strip()):
        return None
    return variant_sig, variant_type


def tc_weaken_hyp(sig: str, type_str: str) -> tuple[str, str] | None:
    return _tc_wrap(weaken_hypothesis_typeclass, sig, type_str)


def tc_strengthen_hyp(sig: str, type_str: str) -> tuple[str, str] | None:
    return _tc_wrap(strengthen_hypothesis_typeclass, sig, type_str)


def tc_strengthen_conc(sig: str, type_str: str) -> tuple[str, str] | None:
    return _tc_wrap(strengthen_conclusion_typeclass, sig, type_str)


def tc_weaken_conc(sig: str, type_str: str) -> tuple[str, str] | None:
    return _tc_wrap(weaken_conclusion_typeclass, sig, type_str)


# ── Bound perturbations ──────────────────────────────────────────────────────
# bound_tighter is the canonical name (matches demo.ipynb). It is implemented
# by the existing perturb_bound function in bounds.py.
bound_tighter = perturb_bound

# ── Registry — keys MUST match _PROPAGATION_RULES above ──────────────────────
TRANSFORMS = {
    "negate":             negate,
    "contrapose":         contrapose,
    "converse":           converse,
    "drop_unused_hyp":    drop_unused_hyp,
    "tc_weaken_hyp":      tc_weaken_hyp,
    "tc_strengthen_hyp":  tc_strengthen_hyp,
    "tc_strengthen_conc": tc_strengthen_conc,
    "tc_weaken_conc":     tc_weaken_conc,
    "flip_bound":         flip_bound,
    "bound_tighter":      bound_tighter,
}

assert set(TRANSFORMS) == set(_PROPAGATION_RULES), (
    "TRANSFORMS keys and _PROPAGATION_RULES keys must match exactly. "
    f"Diff: {set(TRANSFORMS) ^ set(_PROPAGATION_RULES)}"
)

In [18]:
# ── apply_perturbation_chains ─────────────────────────────────────────────────
def apply_perturbation_chains(
    df: pd.DataFrame,
    transforms: dict,
    n_permutations: int = 6,
    random_seed: int = 42,
) -> pd.DataFrame:
    rng = random.Random(random_seed)
    transform_names = list(transforms.keys())
    all_permutations = list(permutations(transform_names))

    results = []

    for _, row in df.iterrows():
        anchor_sig  = row["signature"]
        anchor_type = row["type"]

        k = min(n_permutations, len(all_permutations))
        sampled_permutations = rng.sample(all_permutations, k)

        for perm in sampled_permutations:
            current_sig  = anchor_sig
            current_type = anchor_type
            applied_so_far = []

            for transform_name in perm:
                fn = transforms[transform_name]

                result = fn(current_sig, current_type)
                if result is None:
                    break

                variant_sig, variant_type = result
                if (variant_sig.strip() == current_sig.strip()
                        and variant_type.strip() == current_type.strip()):
                    break

                if not compile_lean(variant_sig, variant_type):
                    break

                # Compiled — save this intermediate as a datapoint
                applied_so_far.append(transform_name)
                results.append({
                    **{k: v for k, v in row.items() if k not in ("signature", "type")},
                    "anchor_signature":      anchor_sig,
                    "anchor_type":           anchor_type,
                    "variant_signature":     variant_sig,
                    "variant_type":          variant_type,
                    "perturbations_applied": list(applied_so_far),
                    "chain_depth":           len(applied_so_far),
                    "is_true":               is_true(list(applied_so_far), variant_sig, variant_type),
                })

                current_sig  = variant_sig
                current_type = variant_type

    result_df = pd.DataFrame(results)
    print(f"Generated {len(result_df):,} compiled variants from {len(df):,} anchors")
    if len(result_df):
        print(result_df["chain_depth"].value_counts().sort_index().to_string())
    return result_df
